<a href="https://colab.research.google.com/github/yanin-dev/mineria-de-datos/blob/main/Copia_de_CONCONCAT_ULT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
from google.colab import drive
# Esto montará tu Google Drive en el directorio '/content/drive'
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [86]:
# Reemplazá estas rutas por las rutas reales de tus archivos
ruta_1 = '/content/drive/MyDrive/mineriadatos2026/dataset-mineria/clientes_crm.csv'
ruta_2 = '/content/drive/MyDrive/mineriadatos2026/dataset-mineria/newsletter.csv'
ruta_3 = '/content/drive/MyDrive/mineriadatos2026/dataset-mineria/soporte.csv'
ruta_4 = '/content/drive/MyDrive/mineriadatos2026/dataset-mineria/ventas.csv'

# Leer los 4 archivos usando read_csv ya que son archivos CSV
df1 = pd.read_csv(ruta_1)
df2 = pd.read_csv(ruta_2)
df3 = pd.read_csv(ruta_3)
df4 = pd.read_csv(ruta_4)

# Verificar la carga
print("clientes_crm:", df1.shape)
print("newsletter:", df2.shape)
print("soporte:", df3.shape)
print("ventas:", df4.shape)

clientes_crm: (433, 13)
newsletter: (325, 6)
soporte: (566, 7)
ventas: (1626, 9)


In [87]:
from pathlib import Path

def buscar_csv(comienzo_del_nombre):
    encontrados = list(Path('.').glob(comienzo_del_nombre + '*.csv'))
    if len(encontrados) == 0:
        raise FileNotFoundError('No se encontró: ' + comienzo_del_nombre)
    return encontrados[0]

In [88]:
import pandas as pd

In [89]:
crm = pd.read_csv(buscar_csv('clientes_crm'))
ventas = pd.read_csv(buscar_csv('ventas'))
soporte = pd.read_csv(buscar_csv('soporte'))
newsletter = pd.read_csv(buscar_csv('newsletter'))
# zonas = pd.read_csv(buscar_csv('zonas_caba'))
# distancias = pd.read_csv(buscar_csv('distancias_sedes'))

print('Los seis archivos fueron cargados correctamente.')

Los seis archivos fueron cargados correctamente.


In [90]:
import unicodedata
import re

def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return None

    nombre = nombre.lower().strip()

    nombre = unicodedata.normalize("NFD", nombre)
    nombre = "".join(
        letra for letra in nombre
        if unicodedata.category(letra) != "Mn"
    )

    nombre = re.sub(r"\s+", " ", nombre)

    return nombre

def limpiar_dni(valor):

    if pd.isna(valor):
        return pd.NA

    cont = 0
    valor = str(valor)
    new_valor = ""

    for i in valor:

        if i.isdigit():
            cont += 1
            new_valor += i

    if cont == 8:
        return new_valor
    else:
        return pd.NA

def limpiar_email(valor): #a@@a.com -> a@a.com
    if pd.isna(valor):
        return pd.NA

    email = str(valor).strip().lower()

    while "@@" in email:
      email = email.replace("@@", "@")

    if '@' not in email:
        return pd.NA
    else:
        return email


In [91]:
crm.columns

Index(['crm_id', 'dni', 'nombre', 'email', 'telefono', 'fecha_nacimiento',
       'edad', 'genero', 'provincia', 'codigo_postal', 'segmento',
       'fecha_alta', 'ingreso_mensual'],
      dtype='object')

In [92]:
ventas.columns

Index(['venta_id', 'documento_cliente', 'cliente', 'correo', 'fecha_compra',
       'monto', 'moneda', 'canal', 'cantidad_items'],
      dtype='object')

In [93]:
soporte.columns

Index(['ticket_id', 'documento', 'nombre_cliente', 'email_contacto',
       'fecha_ticket', 'tipo', 'satisfaccion'],
      dtype='object')

In [94]:
newsletter.columns

Index(['suscriptor_id', 'email', 'nombre', 'estado', 'aperturas_90d',
       'clicks_90d'],
      dtype='object')

In [95]:
df_crm = crm.copy()
df_ventas = ventas.copy()
df_soporte = soporte.copy()
df_news = newsletter.copy()

In [96]:
df_ventas.rename(columns={"documento_cliente": "dni", "cliente": "nombre", "correo": "email"}, inplace=True)
df_soporte.rename(columns={"documento": "dni", "nombre_cliente": "nombre", "email_contacto": "email"}, inplace=True)

In [97]:
df_total = pd.concat([df_crm, df_ventas, df_soporte, df_news], ignore_index=True)

In [98]:
df_total.shape

(2950, 27)

In [99]:
pd.set_option('display.max_columns', None)

In [100]:
df_total.head(1)

,crm_id,dni,nombre,email,telefono,fecha_nacimiento,edad,genero,provincia,codigo_postal,segmento,fecha_alta,ingreso_mensual,venta_id,fecha_compra,monto,moneda,canal,cantidad_items,ticket_id,fecha_ticket,tipo,satisfaccion,suscriptor_id,estado,aperturas_90d,clicks_90d
0,CRM0001,44-232-252,Sebastian Medina,sebastian.medina.32@outlook.com,1162280334,10/05/1962,64.0,H,CABA,C1428,PyME,05/09/2024,256744.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
df_total["nombre_limpio"] = df_total["nombre"].apply(limpiar_nombre)
df_total["dni_limpio"] = df_total["dni"].apply(limpiar_dni)
df_total["email_limpio"] = df_total["email"].apply(limpiar_email)

In [102]:
df_total.shape

(2950, 30)

In [103]:
df_total = df_total.drop_duplicates(subset=["dni_limpio", "email_limpio", "nombre_limpio"])

In [104]:
df_total.shape

(1299, 30)

In [105]:
df_total.isnull().sum()

,0
crm_id,869
dni,411
nombre,0
email,218
telefono,889
fecha_nacimiento,869
edad,869
genero,874
provincia,869
codigo_postal,869


In [106]:
df_total_con_dni = df_total[df_total["dni_limpio"].notna()].copy()

In [107]:
df_total_con_dni.isnull().sum()

,0
crm_id,462
dni,0
nombre,0
email,185
telefono,480
fecha_nacimiento,462
edad,462
genero,467
provincia,462
codigo_postal,462


In [108]:
df_total_con_dni.shape

(879, 30)

In [109]:
df_total_con_dni["dni_limpio"].nunique()

498

In [110]:
nombres_por_dni = df_total_con_dni.groupby("dni_limpio")["nombre_limpio"].apply(list)

In [111]:
nombres_por_dni

,nombre_limpio
dni_limpio,
20003195,[agustina castro]
20084930,[carolina rojas]
20179048,"[micaela acosta, micaela acosta]"
20315891,[andres benitez]
20427454,[nicolas rojas]
...,...
49803465,"[sofia ruiz, sofia federico ruiz, sofia federi..."
49806301,"[camila silva, camila silva]"
49827543,"[carolina gomez, carolina gomez]"


In [112]:
emails_por_dni = df_total_con_dni.groupby("dni_limpio")["email_limpio"].apply(list)

In [113]:
emails_por_dni

,email_limpio
dni_limpio,
20003195,[agustina.castro.90@hotmail.com]
20084930,[carolina.rojas.43@outlook.com]
20179048,"[micaela.acosta.82@hotmail.com, <NA>]"
20315891,[andres.benitez.68@hotmail.com]
20427454,[nicolas.rojas.15@gmail.com]
...,...
49803465,"[sofia.federico.ruiz.81@gmail.com, sofiafederi..."
49806301,"[camila.silva.30@gmail.com, <NA>]"
49827543,"[carolina.gomez.46@outlook.com, <NA>]"


In [114]:
df_total_con_dni["nombres_asociados"] = (df_total_con_dni["dni_limpio"].map(nombres_por_dni))
df_total_con_dni["emails_asociados"] = (df_total_con_dni["dni_limpio"].map(emails_por_dni))

In [115]:
df_total_con_dni.shape

(879, 32)

In [116]:
df_total_con_dni = df_total_con_dni.drop_duplicates(subset=["dni_limpio"])

In [117]:
df_total_con_dni.shape

(498, 32)

In [118]:
df_total_sin_dni_con_email = df_total[df_total["dni_limpio"].isna() & ~df_total["email_limpio"].isna()].copy()

In [119]:
df_total_sin_dni_con_email.isnull().sum()

,0
crm_id,373
dni,377
nombre,0
email,0
telefono,375
fecha_nacimiento,373
edad,373
genero,373
provincia,373
codigo_postal,373


In [120]:
df_total_sin_dni_con_email.shape

(386, 30)

In [121]:
df_total_sin_dni_con_email = df_total_sin_dni_con_email[~df_total_sin_dni_con_email["email_limpio"].isin(df_total_con_dni["email_limpio"].dropna())]

In [122]:
df_total_sin_dni_con_email.shape

(56, 30)

In [123]:
emails_asociados_explode = df_total_con_dni["emails_asociados"].dropna().explode()

In [124]:
df_total_sin_dni_con_email = df_total_sin_dni_con_email[~df_total_sin_dni_con_email["email_limpio"].isin(emails_asociados_explode.dropna())]

In [125]:
df_total_sin_dni_con_email.shape

(31, 30)

In [126]:
df_total_final = pd.concat([df_total_con_dni, df_total_sin_dni_con_email], ignore_index=True)

In [127]:
df_total_final.shape

(529, 32)

In [128]:
df_total_final.head(1)

,crm_id,dni,nombre,email,telefono,fecha_nacimiento,edad,genero,provincia,codigo_postal,segmento,fecha_alta,ingreso_mensual,venta_id,fecha_compra,monto,moneda,canal,cantidad_items,ticket_id,fecha_ticket,tipo,satisfaccion,suscriptor_id,estado,aperturas_90d,clicks_90d,nombre_limpio,dni_limpio,email_limpio,nombres_asociados,emails_asociados
0,CRM0001,44-232-252,Sebastian Medina,sebastian.medina.32@outlook.com,1162280334,10/05/1962,64.0,H,CABA,C1428,PyME,05/09/2024,256744.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sebastian medina,44232252,sebastian.medina.32@outlook.com,"[sebastian medina, sebastian medina]","[sebastian.medina.32@outlook.com, <NA>]"


In [129]:
df_total_sin_dni_sin_email = df_total[df_total["dni_limpio"].isna() & df_total["email_limpio"].isna()].copy()

In [130]:
df_total_sin_dni_sin_email.shape

(34, 30)

In [131]:
df_total_sin_dni_sin_email = df_total_sin_dni_sin_email.drop_duplicates()

In [132]:
df_total_sin_dni_sin_email.shape

(34, 30)

In [133]:
df_total_final = df_total_final.sort_values(["dni_limpio", "email_limpio"])

In [134]:
df_total_final["cliente_id"] = [i for i in range(1,len(df_total_final)+1)]

In [135]:
df_total_final.head(5)

,crm_id,dni,nombre,email,telefono,fecha_nacimiento,edad,genero,provincia,codigo_postal,segmento,fecha_alta,ingreso_mensual,venta_id,fecha_compra,monto,moneda,canal,cantidad_items,ticket_id,fecha_ticket,tipo,satisfaccion,suscriptor_id,estado,aperturas_90d,clicks_90d,nombre_limpio,dni_limpio,email_limpio,nombres_asociados,emails_asociados,cliente_id
116,CRM0123,20-003-195,Agustina Castro,agustina.castro.90@hotmail.com,1168172895,1949-03-18,77.0,f,CABA,C1425,PyME,21-10-24,528903.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,agustina castro,20003195,agustina.castro.90@hotmail.com,[agustina castro],[agustina.castro.90@hotmail.com],1
284,CRM0294,20084930,CAROLINA ROJAS,CAROLINA.ROJAS.43@OUTLOOK.COM,1168929949,21/08/1980,46.0,f,Entre Ríos,E3100,Particular,11/08/2023,499371.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,carolina rojas,20084930,carolina.rojas.43@outlook.com,[carolina rojas],[carolina.rojas.43@outlook.com],2
169,CRM0177,20-179-048,Micaela Acosta,micaela.acosta.82@hotmail.com,1160551066,10/09/1970,55.0,f,Buenos Aires,E3100,Empresa,20/03/2022,379683.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,micaela acosta,20179048,micaela.acosta.82@hotmail.com,"[micaela acosta, micaela acosta]","[micaela.acosta.82@hotmail.com, <NA>]",3
290,CRM0300,20315891,ANDRES BENITEZ,andres.benitez.68@hotmail.com,1137355219,17-11-07,18.0,Masculino,CABA,C1431,Particular,24-07-24,735737.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,andres benitez,20315891,andres.benitez.68@hotmail.com,[andres benitez],[andres.benitez.68@hotmail.com],4
190,CRM0198,20427454,Nicolas Rojas,nicolas.rojas.15@gmail.com,NaN,1971-12-28,54.0,D,Córdoba,E3100,Particular,14-03-25,570242.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nicolas rojas,20427454,nicolas.rojas.15@gmail.com,[nicolas rojas],[nicolas.rojas.15@gmail.com],5


In [136]:
df_total_final.insert(0, "cliente_id", df_total_final.pop("cliente_id"))

In [137]:
df_total_final.tail()

,cliente_id,crm_id,dni,nombre,email,telefono,fecha_nacimiento,edad,genero,provincia,codigo_postal,segmento,fecha_alta,ingreso_mensual,venta_id,fecha_compra,monto,moneda,canal,cantidad_items,ticket_id,fecha_ticket,tipo,satisfaccion,suscriptor_id,estado,aperturas_90d,clicks_90d,nombre_limpio,dni_limpio,email_limpio,nombres_asociados,emails_asociados
516,525,NaN,NaN,Tomas A. Castro,tomasandrescastro35@hotmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0102,rebote,6.0,3.0,tomas a. castro,<NA>,tomasandrescastro35@hotmail.com,NaN,NaN
503,526,NaN,NaN,Tomas A. Torres,tomasandrestorres70@hotmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T00383,05-07-26,Consulta,1.0,NaN,NaN,NaN,NaN,tomas a. torres,<NA>,tomasandrestorres70@hotmail.com,NaN,NaN
528,527,NaN,NaN,Tomas Romero,tomasromero76@outlook.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0320,activo,8.0,1.0,tomas romero,<NA>,tomasromero76@outlook.com,NaN,NaN
520,528,NaN,NaN,Tomas Silva,tomassilva81@empresa.com.ar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0162,activo,2.0,1.0,tomas silva,<NA>,tomassilva81@empresa.com.ar,NaN,NaN
501,529,NaN,NaN,Valentina Benitez,valentina.benitez.52@outlook.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,V001440,20-08-25,20174.93,ARS,Tienda,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,valentina benitez,<NA>,valentina.benitez.52@outlook.com,NaN,NaN


In [138]:
df_total_final.reset_index(drop=True, inplace=True)

In [139]:
df_total_final

,cliente_id,crm_id,dni,nombre,email,telefono,fecha_nacimiento,edad,genero,provincia,codigo_postal,segmento,fecha_alta,ingreso_mensual,venta_id,fecha_compra,monto,moneda,canal,cantidad_items,ticket_id,fecha_ticket,tipo,satisfaccion,suscriptor_id,estado,aperturas_90d,clicks_90d,nombre_limpio,dni_limpio,email_limpio,nombres_asociados,emails_asociados
0,1,CRM0123,20-003-195,Agustina Castro,agustina.castro.90@hotmail.com,1168172895,1949-03-18,77.0,f,CABA,C1425,PyME,21-10-24,528903.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,agustina castro,20003195,agustina.castro.90@hotmail.com,[agustina castro],[agustina.castro.90@hotmail.com]
1,2,CRM0294,20084930,CAROLINA ROJAS,CAROLINA.ROJAS.43@OUTLOOK.COM,1168929949,21/08/1980,46.0,f,Entre Ríos,E3100,Particular,11/08/2023,499371.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,carolina rojas,20084930,carolina.rojas.43@outlook.com,[carolina rojas],[carolina.rojas.43@outlook.com]
2,3,CRM0177,20-179-048,Micaela Acosta,micaela.acosta.82@hotmail.com,1160551066,10/09/1970,55.0,f,Buenos Aires,E3100,Empresa,20/03/2022,379683.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,micaela acosta,20179048,micaela.acosta.82@hotmail.com,"[micaela acosta, micaela acosta]","[micaela.acosta.82@hotmail.com, <NA>]"
3,4,CRM0300,20315891,ANDRES BENITEZ,andres.benitez.68@hotmail.com,1137355219,17-11-07,18.0,Masculino,CABA,C1431,Particular,24-07-24,735737.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,andres benitez,20315891,andres.benitez.68@hotmail.com,[andres benitez],[andres.benitez.68@hotmail.com]
4,5,CRM0198,20427454,Nicolas Rojas,nicolas.rojas.15@gmail.com,NaN,1971-12-28,54.0,D,Córdoba,E3100,Particular,14-03-25,570242.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nicolas rojas,20427454,nicolas.rojas.15@gmail.com,[nicolas rojas],[nicolas.rojas.15@gmail.com]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524,525,NaN,NaN,Tomas A. Castro,tomasandrescastro35@hotmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0102,rebote,6.0,3.0,tomas a. castro,<NA>,tomasandrescastro35@hotmail.com,NaN,NaN
525,526,NaN,NaN,Tomas A. Torres,tomasandrestorres70@hotmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T00383,05-07-26,Consulta,1.0,NaN,NaN,NaN,NaN,tomas a. torres,<NA>,tomasandrestorres70@hotmail.com,NaN,NaN
526,527,NaN,NaN,Tomas Romero,tomasromero76@outlook.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0320,activo,8.0,1.0,tomas romero,<NA>,tomasromero76@outlook.com,NaN,NaN
527,528,NaN,NaN,Tomas Silva,tomassilva81@empresa.com.ar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0162,activo,2.0,1.0,tomas silva,<NA>,tomassilva81@empresa.com.ar,NaN,NaN


In [140]:
for i, j in df_total_final.iterrows():

    # Si emails_asociados no es una lista
    if not isinstance(j["emails_asociados"], list):
        df_total_final.at[i, "emails_asociados"] = []

    # Si hay un email limpio, lo agregamos si todavía no está
    if pd.notna(j["email_limpio"]):
        if j["email_limpio"] not in df_total_final.at[i, "emails_asociados"]:
            df_total_final.at[i, "emails_asociados"].append(j["email_limpio"])

In [141]:
df_total_final

,cliente_id,crm_id,dni,nombre,email,telefono,fecha_nacimiento,edad,genero,provincia,codigo_postal,segmento,fecha_alta,ingreso_mensual,venta_id,fecha_compra,monto,moneda,canal,cantidad_items,ticket_id,fecha_ticket,tipo,satisfaccion,suscriptor_id,estado,aperturas_90d,clicks_90d,nombre_limpio,dni_limpio,email_limpio,nombres_asociados,emails_asociados
0,1,CRM0123,20-003-195,Agustina Castro,agustina.castro.90@hotmail.com,1168172895,1949-03-18,77.0,f,CABA,C1425,PyME,21-10-24,528903.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,agustina castro,20003195,agustina.castro.90@hotmail.com,[agustina castro],[agustina.castro.90@hotmail.com]
1,2,CRM0294,20084930,CAROLINA ROJAS,CAROLINA.ROJAS.43@OUTLOOK.COM,1168929949,21/08/1980,46.0,f,Entre Ríos,E3100,Particular,11/08/2023,499371.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,carolina rojas,20084930,carolina.rojas.43@outlook.com,[carolina rojas],[carolina.rojas.43@outlook.com]
2,3,CRM0177,20-179-048,Micaela Acosta,micaela.acosta.82@hotmail.com,1160551066,10/09/1970,55.0,f,Buenos Aires,E3100,Empresa,20/03/2022,379683.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,micaela acosta,20179048,micaela.acosta.82@hotmail.com,"[micaela acosta, micaela acosta]","[micaela.acosta.82@hotmail.com, <NA>]"
3,4,CRM0300,20315891,ANDRES BENITEZ,andres.benitez.68@hotmail.com,1137355219,17-11-07,18.0,Masculino,CABA,C1431,Particular,24-07-24,735737.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,andres benitez,20315891,andres.benitez.68@hotmail.com,[andres benitez],[andres.benitez.68@hotmail.com]
4,5,CRM0198,20427454,Nicolas Rojas,nicolas.rojas.15@gmail.com,NaN,1971-12-28,54.0,D,Córdoba,E3100,Particular,14-03-25,570242.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nicolas rojas,20427454,nicolas.rojas.15@gmail.com,[nicolas rojas],[nicolas.rojas.15@gmail.com]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524,525,NaN,NaN,Tomas A. Castro,tomasandrescastro35@hotmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0102,rebote,6.0,3.0,tomas a. castro,<NA>,tomasandrescastro35@hotmail.com,NaN,[tomasandrescastro35@hotmail.com]
525,526,NaN,NaN,Tomas A. Torres,tomasandrestorres70@hotmail.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T00383,05-07-26,Consulta,1.0,NaN,NaN,NaN,NaN,tomas a. torres,<NA>,tomasandrestorres70@hotmail.com,NaN,[tomasandrestorres70@hotmail.com]
526,527,NaN,NaN,Tomas Romero,tomasromero76@outlook.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0320,activo,8.0,1.0,tomas romero,<NA>,tomasromero76@outlook.com,NaN,[tomasromero76@outlook.com]
527,528,NaN,NaN,Tomas Silva,tomassilva81@empresa.com.ar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N0162,activo,2.0,1.0,tomas silva,<NA>,tomassilva81@empresa.com.ar,NaN,[tomassilva81@empresa.com.ar]


In [142]:
df_total_final.isnull().sum()

,0
cliente_id,0
crm_id,112
dni,31
nombre,0
email,22
telefono,130
fecha_nacimiento,112
edad,112
genero,117
provincia,112


In [143]:
for index, i in enumerate(df_total_final["emails_asociados"]):
  lista = []
  for j in i:
    if pd.notna(j) and j not in lista:
      lista.append(j)
  df_total_final.at[index, "emails_asociados"] = lista

In [144]:
emails_conflictivos = []

dicc = {}

for i in df_total_final["emails_asociados"].explode().dropna():
    if i not in dicc.keys():
        dicc[i] = 1
    else:
        dicc[i] += 1

for i, j in dicc.items():
    if j > 1:
        print(i, j)
        emails_conflictivos.append(i)

camila.fernandez.61@empresa.com.ar 2


In [145]:
map_dni = {}

for i, fila in df_total_final.iterrows():
    if pd.notna(fila["dni_limpio"]):
        map_dni[fila["dni_limpio"]] = fila["cliente_id"]

In [146]:
map_dni

{'20003195': 1,
 '20084930': 2,
 '20179048': 3,
 '20315891': 4,
 '20427454': 5,
 '20450620': 6,
 '20456261': 7,
 '20568672': 8,
 '20597611': 9,
 '20655052': 10,
 '20684393': 11,
 '20724070': 12,
 '20743307': 13,
 '20745050': 14,
 '20758224': 15,
 '20800214': 16,
 '20993851': 17,
 '21005720': 18,
 '21005823': 19,
 '21015506': 20,
 '21133253': 21,
 '21235333': 22,
 '21351580': 23,
 '21444100': 24,
 '21637899': 25,
 '21719500': 26,
 '21724284': 27,
 '21747335': 28,
 '21832932': 29,
 '21857772': 30,
 '21965745': 31,
 '21980737': 32,
 '22186368': 33,
 '22264345': 34,
 '22370563': 35,
 '22384118': 36,
 '22496371': 37,
 '22512916': 38,
 '22536642': 39,
 '22600186': 40,
 '22710041': 41,
 '22799665': 42,
 '22856150': 43,
 '22919872': 44,
 '22931184': 45,
 '23085199': 46,
 '23159543': 47,
 '23168551': 48,
 '23178977': 49,
 '23347404': 50,
 '23357600': 51,
 '23457785': 52,
 '23469755': 53,
 '23508682': 54,
 '23535325': 55,
 '23547324': 56,
 '23592131': 57,
 '23605706': 58,
 '23723797': 59,
 '2374

In [147]:
map_email = {}

for i, fila in df_total_final.iterrows():
    for email in fila["emails_asociados"]:
        if email not in emails_conflictivos:
            map_email[email] = fila["cliente_id"]

In [148]:
map_email

{'agustina.castro.90@hotmail.com': 1,
 'carolina.rojas.43@outlook.com': 2,
 'micaela.acosta.82@hotmail.com': 3,
 'andres.benitez.68@hotmail.com': 4,
 'nicolas.rojas.15@gmail.com': 5,
 'nicolas.franco.fernandez.26@outlook.com': 6,
 'andres.ramirez.83@empresa.com.ar': 7,
 'andresramirez83@empresa.com.ar': 7,
 'tomas.daniela.martinez.12@gmail.com': 8,
 'juan.flores.26@outlook.com': 9,
 'juanflores26@outlook.com': 9,
 'carolina.gomez.7@empresa.com.ar': 10,
 'maria.ramirez.46@empresa.com.ar': 11,
 'mariaramirez46@empresa.com.ar': 11,
 'daniela.laura.rojas.70@hotmail.com': 12,
 'tomas.rojas.38@gmail.com': 13,
 'nicolas.martin.alvarez.55@gmail.com': 14,
 'paula.rojas.35@hotmail.com': 15,
 'paularojas35@hotmail.com': 15,
 'camila.diaz.18@gmail.com': 16,
 'franco.tomas.ortiz.78@empresa.com.ar': 17,
 'carolina.rodriguez.40@empresa.com.ar': 18,
 'juan.gomez.31@gmail.com': 19,
 'valentina.ramirez.37@hotmail.com': 20,
 'natalia.rojas.15@hotmail.com': 21,
 'nataliarojas15@hotmail.com': 21,
 'franco.

### Ventas

In [149]:
df_ventas

,venta_id,dni,nombre,email,fecha_compra,monto,moneda,canal,cantidad_items
0,V000001,30387900,Pablo Flores,PABLO.PAULA.FLORES.85@HOTMAIL.COM,2025-07-24,64359.88,ARS,App,2
1,V000002,30.387.900,Pablo Flores,pablo.paula.flores.85@hotmail.com,09-03-24,111979.68,ARS,App,5
2,V000003,30-387-900,Pablo Paula Flores,pablo.paula.flores.85@hotmail.com,18/08/2024,16398.76,ARS,Web,2
3,V000004,30387900,Pablo Paula Flores,pablopaulaflores85@hotmail.com,10/05/2025,13055.74,ARS,Marketplace,2
4,V000005,27-131-486,Micaela Martinez,micaela.martinez.28@empresa.com.ar,03/08/2025,13676.59,ARS,App,3
...,...,...,...,...,...,...,...,...,...
1621,V001622,NaN,Martin Silva,martin.silva.2@outlook.com,2025-04-16,53683.58,ARS,Web,5
1622,V001623,37.864.066,Martin Silva,martin.silva.2@outlook.com,2024-12-06,30448.77,ARS,Web,3
1623,V001624,22.856.150,Leandro Ruiz,leandro.ruiz.2@outlook.com,2026-08-15,4800000.00,ARS,Web,86
1624,V001625,42.995.931,Julieta Rodríguez,julieta.rodriguez.21@gmail.com,2026-08-15,4800000.00,ARS,Web,86


In [150]:
df_ventas["dni_limpio"] = df_ventas["dni"].apply(limpiar_dni)
df_ventas["email_limpio"] = df_ventas["email"].apply(limpiar_email)

In [151]:
df_ventas

,venta_id,dni,nombre,email,fecha_compra,monto,moneda,canal,cantidad_items,dni_limpio,email_limpio
0,V000001,30387900,Pablo Flores,PABLO.PAULA.FLORES.85@HOTMAIL.COM,2025-07-24,64359.88,ARS,App,2,30387900,pablo.paula.flores.85@hotmail.com
1,V000002,30.387.900,Pablo Flores,pablo.paula.flores.85@hotmail.com,09-03-24,111979.68,ARS,App,5,30387900,pablo.paula.flores.85@hotmail.com
2,V000003,30-387-900,Pablo Paula Flores,pablo.paula.flores.85@hotmail.com,18/08/2024,16398.76,ARS,Web,2,30387900,pablo.paula.flores.85@hotmail.com
3,V000004,30387900,Pablo Paula Flores,pablopaulaflores85@hotmail.com,10/05/2025,13055.74,ARS,Marketplace,2,30387900,pablopaulaflores85@hotmail.com
4,V000005,27-131-486,Micaela Martinez,micaela.martinez.28@empresa.com.ar,03/08/2025,13676.59,ARS,App,3,27131486,micaela.martinez.28@empresa.com.ar
...,...,...,...,...,...,...,...,...,...,...,...
1621,V001622,NaN,Martin Silva,martin.silva.2@outlook.com,2025-04-16,53683.58,ARS,Web,5,<NA>,martin.silva.2@outlook.com
1622,V001623,37.864.066,Martin Silva,martin.silva.2@outlook.com,2024-12-06,30448.77,ARS,Web,3,37864066,martin.silva.2@outlook.com
1623,V001624,22.856.150,Leandro Ruiz,leandro.ruiz.2@outlook.com,2026-08-15,4800000.00,ARS,Web,86,22856150,leandro.ruiz.2@outlook.com
1624,V001625,42.995.931,Julieta Rodríguez,julieta.rodriguez.21@gmail.com,2026-08-15,4800000.00,ARS,Web,86,42995931,julieta.rodriguez.21@gmail.com


In [152]:
# 1. DNI
df_ventas["cliente_id"] = df_ventas["dni_limpio"].map(map_dni)

por_dni = df_ventas["cliente_id"].notna().sum() # 1496

# 2. Email solamente para los que fallaron por DNI
sin_id = df_ventas["cliente_id"].isna()

df_ventas.loc[sin_id, "cliente_id"] = (df_ventas.loc[sin_id, "email_limpio"].map(map_email))

total = df_ventas["cliente_id"].notna().sum() # 1608
por_email = total - por_dni # 112

print("Por DNI:", por_dni)
print("Por email:", por_email)
print("Sin identificar:", df_ventas["cliente_id"].isna().sum())

Por DNI: 1496
Por email: 112
Sin identificar: 18


In [153]:
df_ventas

,venta_id,dni,nombre,email,fecha_compra,monto,moneda,canal,cantidad_items,dni_limpio,email_limpio,cliente_id
0,V000001,30387900,Pablo Flores,PABLO.PAULA.FLORES.85@HOTMAIL.COM,2025-07-24,64359.88,ARS,App,2,30387900,pablo.paula.flores.85@hotmail.com,171.0
1,V000002,30.387.900,Pablo Flores,pablo.paula.flores.85@hotmail.com,09-03-24,111979.68,ARS,App,5,30387900,pablo.paula.flores.85@hotmail.com,171.0
2,V000003,30-387-900,Pablo Paula Flores,pablo.paula.flores.85@hotmail.com,18/08/2024,16398.76,ARS,Web,2,30387900,pablo.paula.flores.85@hotmail.com,171.0
3,V000004,30387900,Pablo Paula Flores,pablopaulaflores85@hotmail.com,10/05/2025,13055.74,ARS,Marketplace,2,30387900,pablopaulaflores85@hotmail.com,171.0
4,V000005,27-131-486,Micaela Martinez,micaela.martinez.28@empresa.com.ar,03/08/2025,13676.59,ARS,App,3,27131486,micaela.martinez.28@empresa.com.ar,115.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1621,V001622,NaN,Martin Silva,martin.silva.2@outlook.com,2025-04-16,53683.58,ARS,Web,5,<NA>,martin.silva.2@outlook.com,310.0
1622,V001623,37.864.066,Martin Silva,martin.silva.2@outlook.com,2024-12-06,30448.77,ARS,Web,3,37864066,martin.silva.2@outlook.com,310.0
1623,V001624,22.856.150,Leandro Ruiz,leandro.ruiz.2@outlook.com,2026-08-15,4800000.00,ARS,Web,86,22856150,leandro.ruiz.2@outlook.com,43.0
1624,V001625,42.995.931,Julieta Rodríguez,julieta.rodriguez.21@gmail.com,2026-08-15,4800000.00,ARS,Web,86,42995931,julieta.rodriguez.21@gmail.com,392.0


In [154]:
#df_ventas = df_ventas[["cliente_id"] + [col for col in df_ventas.columns if col != "cliente_id"]]
df_ventas["cliente_id"] = df_ventas["cliente_id"].astype("Int64")
df_ventas.head()

,venta_id,dni,nombre,email,fecha_compra,monto,moneda,canal,cantidad_items,dni_limpio,email_limpio,cliente_id
0,V000001,30387900,Pablo Flores,PABLO.PAULA.FLORES.85@HOTMAIL.COM,2025-07-24,64359.88,ARS,App,2,30387900,pablo.paula.flores.85@hotmail.com,171
1,V000002,30.387.900,Pablo Flores,pablo.paula.flores.85@hotmail.com,09-03-24,111979.68,ARS,App,5,30387900,pablo.paula.flores.85@hotmail.com,171
2,V000003,30-387-900,Pablo Paula Flores,pablo.paula.flores.85@hotmail.com,18/08/2024,16398.76,ARS,Web,2,30387900,pablo.paula.flores.85@hotmail.com,171
3,V000004,30387900,Pablo Paula Flores,pablopaulaflores85@hotmail.com,10/05/2025,13055.74,ARS,Marketplace,2,30387900,pablopaulaflores85@hotmail.com,171
4,V000005,27-131-486,Micaela Martinez,micaela.martinez.28@empresa.com.ar,03/08/2025,13676.59,ARS,App,3,27131486,micaela.martinez.28@empresa.com.ar,115


In [155]:
df_ventas.columns

Index(['venta_id', 'dni', 'nombre', 'email', 'fecha_compra', 'monto', 'moneda',
       'canal', 'cantidad_items', 'dni_limpio', 'email_limpio', 'cliente_id'],
      dtype='object')

In [156]:
df_ventas[["venta_id", "fecha_compra", "monto", "moneda", "canal", "cantidad_items"]].isna().sum()

,0
venta_id,0
fecha_compra,0
monto,0
moneda,0
canal,0
cantidad_items,0


In [157]:
df_ventas.describe().T

,count,mean,std,min,25%,50%,75%,max
monto,1626.0,61380.133782,216544.564105,8.96,17917.315,31629.22,59329.7,4800000.0
cantidad_items,1626.0,2.466175,3.822737,1.0,1.0,2.0,3.0,86.0
cliente_id,1608.0,249.31592,148.004721,1.0,118.0,246.0,380.0,529.0


In [158]:
df_ventas["moneda"].unique()

array(['ARS', 'USD'], dtype=object)

In [159]:
df_ventas.groupby("moneda")["monto"].describe()

,count,mean,std,min,25%,50%,75%,max
moneda,,,,,,,,
ARS,1608.0,62066.758924,217656.072074,2500.00,18209.175,32169.34,60131.7275,4800000.00
USD,18.0,41.621111,61.109652,8.96,14.245,17.90,36.3975,262.34


In [160]:
df_ventas["moneda"].value_counts(dropna=False)


,count
moneda,
ARS,1608
USD,18


In [161]:
df_ventas["fecha_compra"].head(10)

,fecha_compra
0,2025-07-24
1,09-03-24
2,18/08/2024
3,10/05/2025
4,03/08/2025
5,2025-08-11
6,2024-03-10
7,04/03/2024
8,05-05-26
9,2025-07-16


#### Se nos presenta el siguiente problema: Las fechas no están normalizadas, y hay pocos montos en dolares que deberíamos pasar a pesos, pero antes de hacer la conversión debemos establecer a que precio estaba el dolar en el momento de la compra (aproximadamente)